<a href="https://colab.research.google.com/github/harsha475/AI-Learning_Codebook/blob/main/Copy_of_Transformer_example_program_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# ==========================================
# Basic Transformer: Next Word Prediction
# ==========================================

sentence = """
artificial intelligence is transforming the world by enabling
machines to learn from data make decisions solve problems
and assist humans in many different tasks every day
"""

# Convert text into words
words = sentence.lower().split()

# ------------------------------------------
# Build Vocabulary
# ------------------------------------------

vocab = sorted(set(words))

word_to_id = {word: idx for idx, word in enumerate(vocab)}
id_to_word = {idx: word for word, idx in word_to_id.items()}

vocab_size = len(vocab)

print("Vocabulary Size:", vocab_size)

# ------------------------------------------
# Create Input-Output Pairs
# ------------------------------------------

inputs = []
targets = []

for i in range(len(words) - 1):
    inputs.append(word_to_id[words[i]])
    targets.append(word_to_id[words[i + 1]])

inputs = torch.tensor(inputs).unsqueeze(0)
targets = torch.tensor(targets)

# ------------------------------------------
# Transformer Components
# ------------------------------------------

embedding = nn.Embedding(
    num_embeddings=vocab_size,
    embedding_dim=64
)

encoder_layer = nn.TransformerEncoderLayer(
    d_model=64,
    nhead=4,
    batch_first=True
)

transformer = nn.TransformerEncoder(
    encoder_layer,
    num_layers=2
)

output_layer = nn.Linear(
    64,
    vocab_size
)

# ------------------------------------------
# Optimizer & Loss Function
# ------------------------------------------

parameters = (
    list(embedding.parameters()) +
    list(transformer.parameters()) +
    list(output_layer.parameters())
)

optimizer = optim.Adam(parameters, lr=0.01)

loss_function = nn.CrossEntropyLoss()

# ------------------------------------------
# Training
# ------------------------------------------

print("\nTraining Started...\n")

for epoch in range(500):

    optimizer.zero_grad()

    embedded_words = embedding(inputs)

    transformer_output = transformer(
        embedded_words
    )

    predictions = output_layer(
        transformer_output
    )

    loss = loss_function(
        predictions.squeeze(0),
        targets
    )

    loss.backward()
    optimizer.step()

    if epoch % 100 == 0:
        print(
            f"Epoch {epoch:3d} | Loss = {loss.item():.4f}"
        )

# ------------------------------------------
# Test Prediction
# ------------------------------------------

test_word = "artificial"

test_input = torch.tensor(
    [[word_to_id[test_word]]]
)

with torch.no_grad():

    embedded_test = embedding(test_input)

    transformer_result = transformer(
        embedded_test
    )

    output = output_layer(
        transformer_result
    )

    predicted_index = torch.argmax(
        output,
        dim=-1
    )

predicted_word = id_to_word[
    predicted_index.item()
]

# ------------------------------------------
# Final Output
# ------------------------------------------

print("\n" + "-" * 40)
print("TRANSFORMER PREDICTION RESULT")
print("-" * 40)
print(f"Input Word      : {test_word}")
print(f"Predicted Word  : {predicted_word}")
print("-" * 40)

Vocabulary Size: 26

Training Started...

Epoch   0 | Loss = 3.4700
Epoch 100 | Loss = 0.0008
Epoch 200 | Loss = 0.0004
Epoch 300 | Loss = 0.0002
Epoch 400 | Loss = 0.0002

----------------------------------------
TRANSFORMER PREDICTION RESULT
----------------------------------------
Input Word      : artificial
Predicted Word  : intelligence
----------------------------------------
